In [33]:
import sys
sys.path.append("../src")
import pandas as pd
from utils.front_model import openai_generate
from utils.utils import validate_json
from collections import Counter

In [34]:
df = pd.read_excel("../data/ai_lab_1.xlsx")

In [35]:
def make_cot_prompt(property_id: str, input: str) -> str:
    pid = str(property_id)
    return f"""
You are a deterministic JSON extraction engine.

Think step-by-step internally before answering.
Do NOT output your reasoning.

Return EXACTLY one JSON object and NOTHING else.

Output format:
{{ "{pid}": number | null }}

Task:
Extract the total land plot area.

Allowed units:
- m2, sqm, square meters
- sq ft, sqft, ft2
- hectares (ha)

Conversions:
- 1 hectare = 10000 square meters
- 1 square foot = 0.092903 square meters

Rounding:
Round to exactly 2 decimals.

Internal reasoning steps (do not output):
1. Scan the text for numbers associated with land/plot area.
2. Ignore building area, living area, perimeter, distances, and heights.
3. Identify the unit (m2, sqft, ha).
4. Normalize the number (remove commas if needed).
5. Convert to square meters if required.
6. Round to exactly 2 decimals.
7. Output JSON.

If no valid land plot area exists:
{{ "{pid}": null }}


<text>
{input}
</text>
""".strip()

In [ ]:
count = 1

def generate(prompt):
    res = []
    for _ in range(count):
        output = openai_generate(prompt)
        res.append(output)

    filtered = [p for p in res if p is not None]
    return Counter(filtered).most_common(1)[0][0]

In [37]:
def create_ground_truth():
    results = []
    for _, row in df.iterrows():
        pid = str(row["property_id"])
        prompt = make_cot_prompt(pid, row["property_description"])
        output = generate(prompt)
        val, _, _ = validate_json(pid, output)
        results.append({
            "pid": pid,
            "output": val
        })
    return pd.DataFrame(results)

In [38]:
res = create_ground_truth()

{ "1": 30000.00 }
{ "2": 750000.00 }
{ "3": 70000.00 }
{ "4": 60702.85 }
{ "5": 200000.00 }
{ "6": 100000.00 }
{ "7": 52609.13 }
{ "8": 6503.21 }
{ "9": 4180.63 }
{ "10": 25000.00 }
{ "11": 18580.60 }
{ "12": null }
{ "13": 5574.18 }
{ "14": 46451.50 }
{ "15": 3251.60 }
{ "16": 4645.15 }
{ "17": 1000000.00 }
{ "18": 27870.90 }
{ "19": 12000.00 }
{ "20": null }
{ "21": null }
{ "22": 7500.00 }
{"23": 50000.00}
{ "24": 30000.00 }
{ "25": 40000.00 }
{ "26": 25000.00 }
{ "27": 1000000.00 }
{ "28": 18210.87 }
{ "29": 20000.00 }
{ "30": 12140589.05 }
{ "31": 14163.65 }
{ "32": 3716.12 }
{ "33": 9290.30 }
{ "34": 9290.30 }
{ "35": 141639.97 }
{ "36": 5574.18 }
{ "37": 6967.73 }
{ "38": 2787.09 }
{ "39": 18000.00 }
{ "40": 27870.90 }
{ "41": 32374.85 }
{ "42": 8093.72 }
{ "43": 1858.06 }
{ "44": 8000.00 }
{ "45": 1147.48 }
{ "46": 232.26 }
{ "47": 1500.00 }
{ "48": 2500.00 }
{ "49": 7500.00 }
{ "50": 464.52 }
{ "51": 5000.00 }
{ "52": 8000.00 }
{ "53": 12000.00 }
{ "54": 15000.00 }
{ "55": 100

In [39]:
res.to_csv("../data/ground_truth_new.csv")